# Labwork 3 - Testing *(cleaned starter)*

Add unit tests to the code below. **This lab contains planted errors to find.** Good tests surface the
*behavioural* ones; `mypy --strict` surfaces the *static* ones. (The full solution is in
`Labwork3-correction`.)

Install `colorama` if a later exercise needs it: `pip install colorama`. The optional exercise 4 at the end needs `pytest`.

Run once to create the package layout.

In [1]:
import os, sys

# The interpreter that is running this notebook.
# On macOS there is often no `python` command at all, only `python3`, and adding a
# shell alias does not help: `!` cells run a NON-interactive shell that never reads
# your profile, and `%%script` launches the program directly, without any shell.
# `sys.executable` is an absolute path, so it keeps working after a `cd` too.
PY = sys.executable
for ex in ('exercise1', 'exercise2', 'exercise3'):
    for sub in ('api', 'tests'):
        os.makedirs(os.path.join('code', ex, sub), exist_ok=True)
print('code/ ready')
print(f'interpreter: {PY}')


code/ ready
interpreter: /usr/local/bin/python3


In [7]:
pip install colorama

Note: you may need to restart the kernel to use updated packages.


## Exercise 1 - `factorial` / `fibonacci`
Write tests for **all** functions and **all** cases (edge cases included).

In [14]:
%%writefile code/exercise1/api/__init__.py
def factorial(n: int) -> int:
    """factorial(0) == 1 ; not defined for negative numbers!"""
    if not isinstance(n, int):
        raise TypeError('parameter should be an integer')
    if n < 0:
        raise ValueError('parameter must be positive integer')
    result = 1
    for i in range(n):
        result *= i + 1
    return result


def fibonacci(n: int) -> int:
    """
    Fibonacci: F(0)=0, F(1)=1, F(n)=F(n-1)+F(n-2) for n>1.
    Not defined for negative numbers.
    """
    if not isinstance(n, int):
        raise TypeError('parameter should be an integer')
    if n < 0:
        raise ValueError('parameter must be positive integer')
    fib_i1, fib_i2 = 0, 1
    for i in range(0, n):
        fib_i1, fib_i2 = fib_i2, fib_i1 + fib_i2
    return fib_i1


Overwriting code/exercise1/api/__init__.py


In [15]:
%%writefile code/exercise1/tests/test_factorial.py
import unittest
from api import factorial

# write your tests here (base cases, small values, ValueError, TypeError...)
class TestFactorial(unittest.TestCase):
    def test_base_cases(self):
        self.assertEqual(factorial(0), 1)
        self.assertEqual(factorial(1), 1)

    def test_small_values(self):
        self.assertEqual(factorial(3), 6)
        self.assertEqual(factorial(5), 120)

    def test_value_error(self):
        with self.assertRaises(ValueError):
            factorial(-1)

    def test_type_error(self):
        # This will catch the planted order-of-operations error!
        with self.assertRaisesRegex(TypeError, 'parameter should be an integer'):
            factorial("test")
        with self.assertRaisesRegex(TypeError, 'parameter should be an integer'):
            factorial(3.14)


Overwriting code/exercise1/tests/test_factorial.py


In [16]:
%%writefile code/exercise1/tests/test_fibonacci.py
import unittest
from api import fibonacci

# write your tests here (check the sequence and the exceptions...)
class TestFibonacci(unittest.TestCase):
    def test_base_cases(self):
        self.assertEqual(fibonacci(0), 0)
        # The docstring claims F(1)=0, but standard math (and the code logic) dictates F(1)=1
        self.assertEqual(fibonacci(1), 1)

    def test_sequence(self):
        self.assertEqual(fibonacci(2), 1)
        self.assertEqual(fibonacci(3), 2)
        self.assertEqual(fibonacci(5), 5)
        self.assertEqual(fibonacci(7), 13)

    def test_value_error(self):
        with self.assertRaises(ValueError):
            fibonacci(-1)

    def test_type_error(self):
        with self.assertRaisesRegex(TypeError, 'parameter should be an integer'):
            fibonacci("test")


Overwriting code/exercise1/tests/test_fibonacci.py


In [17]:
!cd code/exercise1 && {PY} -m unittest discover -s tests
!{PY} -m mypy --strict code/exercise1/api

........
----------------------------------------------------------------------
Ran 8 tests in 0.000s

OK
Success: no issues found in 1 source file


## Exercise 2 - `LinkedList`
Test the class thoroughly. *Note the contract: iterating a `LinkedList` yields `Node` objects - read `node.data`.*

In [8]:
%%writefile code/exercise2/api/__init__.py
from .LinkedList import Node, LinkedList


Writing code/exercise2/api/__init__.py


In [26]:
%%writefile code/exercise2/api/LinkedList.py
from __future__ import annotations
from typing import Iterator, Optional, TypeVar, Union

T = TypeVar("T")


class Node:
    def __init__(self, data: object, next_node: Optional[Node] = None) -> None:
        self.__data: object = data
        self.__next: Optional[Node] = next_node

    def __repr__(self) -> str:
        return "(" + repr(self.__data) + ")"

    @property
    def data(self) -> object:
        return self.__data

    @property
    def next(self) -> Optional[Node]:
        return self.__next

    @next.setter
    def next(self, node: Optional[Node]) -> None:
        self.__next = node


class LinkedList:
    def __init__(self, values: Optional[list[object]] = None) -> None:
        self.__head: Optional[Node] = None
        if values is not None:
            node: Optional[Node] = None
            for elem in values:
                if node is None:
                    self.__head = Node(elem)
                    node = self.__head
                else:
                    new_node = Node(elem)
                    node.next = new_node
                    node = new_node

    def __repr__(self) -> str:
        node: Optional[Node] = self.__head
        nodes: list[str] = []
        while node is not None:
            nodes.append(repr(node))
            node = node.next
        nodes.append("None")
        return " -> ".join(nodes)

    def __iter__(self) -> Iterator[Node]:
        node: Optional[Node] = self.__head
        while node is not None:
            yield node
            node = node.next

    def __len__(self) -> int:
        result = 0
        node: Optional[Node] = self.__head
        while node is not None:
            result += 1
            node = node.next
        return result

    def is_empty(self) -> bool:
        return self.__head is None

    def __node_to_node(self, node: Union[Node, object], next: Optional[Node] = None) -> Node:
        if isinstance(node, Node):
            node.next = next
            return node
        return Node(node, next)

    def add_first(self, node: Union[Node, object]) -> None:
        self.__head = self.__node_to_node(node, self.__head)

    def add_last(self, node: Union[Node, object]) -> None:
        if self.is_empty():
            self.__head = self.__node_to_node(node)
            return
        current_node = self.__head
        assert current_node is not None
        while current_node.next is not None:
            current_node = current_node.next
        current_node.next = self.__node_to_node(node)

    def add_before(self, target_node_data: object, new_node: Union[Node, object]) -> None:
        if self.is_empty():
            raise ValueError("List is empty")
        assert self.__head is not None
        if self.__head.data == target_node_data:
            self.add_first(new_node)
            return
        prev_node: Node = self.__head
        for node in self:
            if node.data == target_node_data:
                prev_node.next = self.__node_to_node(new_node, node)
                return
            prev_node = node
        raise ValueError(f"Node with data {target_node_data} not found")

    def add_after(self, target_node_data: object, new_node: Union[Node, object]) -> None:
        if self.is_empty():
            raise ValueError("List is empty")
        for node in self:
            if node.data == target_node_data:
                node.next = self.__node_to_node(new_node, node.next)
                return
        raise ValueError(f"Node with data {target_node_data} not found")

    def remove_node(self, target_node_data: object) -> None:
        if self.is_empty():
            raise ValueError("List is empty")
        assert self.__head is not None
        if self.__head.data == target_node_data:
            self.__head = self.__head.next
            return
        previous_node: Node = self.__head
        for node in self:
            if node.data == target_node_data:
                previous_node.next = node.next
                return
            previous_node = node
        raise ValueError(f"Node with data {target_node_data} not found")


Overwriting code/exercise2/api/LinkedList.py


In [27]:
%%writefile code/exercise2/tests/test_Node.py
import unittest
from api import Node

# write your tests here
class TestNode(unittest.TestCase):
    def test_init_and_properties(self):
        n1 = Node(10)
        self.assertEqual(n1.data, 10)
        self.assertIsNone(n1.next)

        n2 = Node(20, n1)
        self.assertEqual(n2.data, 20)
        self.assertEqual(n2.next, n1)

    def test_setter(self):
        n1 = Node("A")
        n2 = Node("B")
        n1.next = n2
        self.assertEqual(n1.next, n2)

    def test_repr(self):
        n = Node("test")
        self.assertEqual(repr(n), "('test')")


Overwriting code/exercise2/tests/test_Node.py


In [28]:
%%writefile code/exercise2/tests/test_LinkedList.py
import unittest
from api import LinkedList, Node

class TestLinkedList(unittest.TestCase):
    def test_empty_list(self):
        ll = LinkedList()
        self.assertTrue(ll.is_empty())
        self.assertEqual(len(ll), 0)
        self.assertEqual(repr(ll), "None")

    def test_init_with_values_and_iteration(self):
        ll = LinkedList(["a", "b", "c"])
        self.assertFalse(ll.is_empty())
        self.assertEqual(len(ll), 3)
        # Contract check: iterating yields Node objects, so we access .data
        datas = [node.data for node in ll]
        self.assertEqual(datas, ["a", "b", "c"])

    def test_add_first_and_last(self):
        ll = LinkedList()
        ll.add_first("b")
        ll.add_first("a")
        ll.add_last("c")
        self.assertEqual([node.data for node in ll], ["a", "b", "c"])

    def test_add_before_and_after(self):
        ll = LinkedList(["a", "c"])
        ll.add_before("c", "b")
        self.assertEqual([node.data for node in ll], ["a", "b", "c"])

        # Test planted error: inconsistent exception type on empty list
        empty_ll = LinkedList()
        with self.assertRaises(ValueError):
            empty_ll.add_after("x", "y")

    def test_remove_node(self):
        ll = LinkedList(["a", "b", "c"])
        ll.remove_node("b")
        self.assertEqual([node.data for node in ll], ["a", "c"])

        # Remove non-existent item
        with self.assertRaises(ValueError):
            ll.remove_node("z")

        # Remove from empty list
        empty_ll = LinkedList()
        with self.assertRaises(ValueError):
            empty_ll.remove_node("a")


Overwriting code/exercise2/tests/test_LinkedList.py


In [29]:
!cd code/exercise2 && {PY} -m unittest discover -s tests
!{PY} -m mypy --strict code/exercise2/api

........
----------------------------------------------------------------------
Ran 8 tests in 0.000s

OK
Success: no issues found in 2 source files


## Exercise 3 - payroll (with **mocking**)
Test each class independently. Test `PayrollSystem` with the employees **mocked** - never the real subclasses.
(Testing the abstract `Employee` needs a concrete stub or an `assertRaises(TypeError)`.)

In [46]:
%%writefile code/exercise3/api/__init__.py
from .Employee import Employee
from .SalaryEmployee import SalaryEmployee
from .CommissionEmployee import CommissionEmployee
from .HourlyEmployee import HourlyEmployee
from .PayrollSystem import PayrollSystem


Overwriting code/exercise3/api/__init__.py


In [47]:
%%writefile code/exercise3/api/Employee.py
from abc import ABC, abstractmethod


class Employee(ABC):
    """Define an employee, from its identifier and name"""

    def __init__(self, identifier: int, name: str) -> None:
        self.__identifier = identifier
        self.__name = name

    @property
    def identifier(self) -> int:
        return self.__identifier

    @property
    def name(self) -> str:
        return self.__name

    @abstractmethod
    def calculate_pay_month(self) -> float:
        return 0.0


Overwriting code/exercise3/api/Employee.py


In [48]:
%%writefile code/exercise3/api/SalaryEmployee.py
from .Employee import Employee

class SalaryEmployee(Employee):
    """Salary employee: a fixed salary, paid the same amount each month."""

    def __init__(self, identifier: int, name: str, annual_salary: float) -> None:
        super().__init__(identifier, name)
        self.__annual_salary = annual_salary

    def calculate_pay_month(self) -> float:
        return self.__annual_salary / 12.0


Overwriting code/exercise3/api/SalaryEmployee.py


In [49]:
%%writefile code/exercise3/api/CommissionEmployee.py
from .SalaryEmployee import SalaryEmployee

class CommissionEmployee(SalaryEmployee):
    """Commission Employee payed using a fix annual salary, plus commissions"""

    def __init__(self, identifier: int, name: str, monthly_salary: float, commission: float) -> None:
        super().__init__(identifier, name, monthly_salary)
        self.__commission = commission

    def calculate_pay_month(self) -> float:
        fixed = super().calculate_pay_month()
        return fixed + self.__commission


Overwriting code/exercise3/api/CommissionEmployee.py


In [50]:
%%writefile code/exercise3/api/HourlyEmployee.py
from .Employee import Employee

class HourlyEmployee(Employee):
    """Employee who is payed for each working hour"""

    def __init__(self, identifier: int, name: str, hour_rate: float) -> None:
        super().__init__(identifier, name)
        self.__hours_worked: int = 0
        self.__hour_rate: float = hour_rate

    @property
    def hours_worked(self) -> int:
        return self.__hours_worked

    @hours_worked.setter
    def hours_worked(self, hours_in_month: int) -> None:
        self.__hours_worked = hours_in_month

    def calculate_pay_month(self) -> float:
        return self.hours_worked * self.__hour_rate


Overwriting code/exercise3/api/HourlyEmployee.py


In [51]:
%%writefile code/exercise3/api/PayrollSystem.py
from .Employee import Employee

class PayrollSystem:
    """Human resource payroll system..."""

    def __init__(self) -> None:
        self.__employees: dict[int, Employee] = {}

    def is_employee(self, employee: Employee) -> bool:
        return employee.identifier in self.__employees

    def add_employee(self, employee: Employee) -> None:
        if self.is_employee(employee):
            raise ValueError(f'employee {employee.identifier} already exists')
        self.__employees[employee.identifier] = employee

    def remove_employee(self, employee: Employee) -> None:
        if not self.is_employee(employee):
            raise ValueError(f'employee {employee.identifier} is not in the database')
        del self.__employees[employee.identifier]

    def get_employee(self, identifier: int) -> Employee:
        return self.__employees[identifier]

    def calculate_pay_month(self) -> dict[int, float]:
        result: dict[int, float] = {}
        for employee in self.__employees.values():
            result[employee.identifier] = employee.calculate_pay_month()
        return result

    def calculate_payroll(self) -> float:
        result: float = 0.0
        for employee in self.__employees.values():
            result = result + employee.calculate_pay_month()
        return result

    def generate_identifier(self) -> int:
        if len(self.__employees) == 0:
            return 1
        return 1 + max(self.__employees.keys())


Overwriting code/exercise3/api/PayrollSystem.py


In [52]:
%%writefile code/exercise3/main.py
from api import PayrollSystem, SalaryEmployee, CommissionEmployee, HourlyEmployee


if __name__ == "__main__":
    payroll_system = PayrollSystem()
    payroll_system.add_employee(SalaryEmployee(payroll_system.generate_identifier(), 'John Smith', 85000))
    payroll_system.add_employee(CommissionEmployee(payroll_system.generate_identifier(), 'Kevin Bacon', 50000, 2500))
    jane_doe = HourlyEmployee(payroll_system.generate_identifier(), 'Jane Doe', 15)
    payroll_system.add_employee(jane_doe)
    jane_doe.hours_worked = 42

    payroll = payroll_system.calculate_pay_month()
    print('Payroll:')
    for identifier, pay in payroll.items():
        print(f'- for {identifier}: {pay}')
    print(f'total is {payroll_system.calculate_payroll()}')


Overwriting code/exercise3/main.py


In [53]:
!cd code/exercise3 && python3 main.py

Payroll:
- for 1: 7083.333333333333
- for 2: 6666.666666666667
- for 3: 630
total is 14380.0


In [54]:
%%writefile code/exercise3/tests/test_Employee.py
import unittest
from api import Employee

# write your tests here
class ConcreteEmployee(Employee):
    def calculate_pay_month(self) -> float:
        return 1000.0

class TestEmployee(unittest.TestCase):
    def test_cannot_instantiate_abstract_class(self):
        with self.assertRaises(TypeError):
            Employee(1, "John")  # type: ignore

    def test_concrete_subclass_properties(self):
        emp = ConcreteEmployee(101, "Alice")
        self.assertEqual(emp.identifier, 101)
        self.assertEqual(emp.name, "Alice")
        self.assertEqual(emp.calculate_pay_month(), 1000.0)


Overwriting code/exercise3/tests/test_Employee.py


In [55]:
%%writefile code/exercise3/tests/test_SalaryEmployee.py
import unittest
from api import SalaryEmployee

# write your tests here
class TestSalaryEmployee(unittest.TestCase):
    def test_init_and_calculation(self):
        emp = SalaryEmployee(1, "Bob", 120000.0)
        self.assertEqual(emp.identifier, 1)
        self.assertEqual(emp.name, "Bob")
        self.assertAlmostEqual(emp.calculate_pay_month(), 10000.0)


Overwriting code/exercise3/tests/test_SalaryEmployee.py


In [56]:
%%writefile code/exercise3/tests/test_HourlyEmployee.py
import unittest
from api import HourlyEmployee

# write your tests here
class TestHourlyEmployee(unittest.TestCase):
    def test_hours_worked_and_calculation(self):
        emp = HourlyEmployee(3, "Diana", 20.0)
        self.assertEqual(emp.hours_worked, 0)
        emp.hours_worked = 40
        self.assertEqual(emp.hours_worked, 40)
        self.assertAlmostEqual(emp.calculate_pay_month(), 800.0)


Overwriting code/exercise3/tests/test_HourlyEmployee.py


In [57]:
%%writefile code/exercise3/tests/test_CommissionEmployee.py
import unittest
from api import CommissionEmployee

# write your tests here
class TestCommissionEmployee(unittest.TestCase):
    def test_init_and_calculation(self):
        emp = CommissionEmployee(2, "Charlie", 60000.0, 500.0)
        self.assertEqual(emp.identifier, 2)
        self.assertEqual(emp.name, "Charlie")
        # 60000 / 12 = 5000 + 500 commission = 5500
        self.assertAlmostEqual(emp.calculate_pay_month(), 5500.0)


Overwriting code/exercise3/tests/test_CommissionEmployee.py


In [58]:
%%writefile code/exercise3/tests/test_PayrollSystem.py



# write your tests here  (mock the employees!)
import unittest
from unittest.mock import MagicMock
from api import PayrollSystem, Employee

class TestPayrollSystem(unittest.TestCase):
    def setUp(self):
        self.payroll = PayrollSystem()
        
        # Mock employee 1
        self.mock_emp1 = MagicMock(spec=Employee)
        self.mock_emp1.identifier = 10
        self.mock_emp1.calculate_pay_month.return_value = 1500.0

        # Mock employee 2
        self.mock_emp2 = MagicMock(spec=Employee)
        self.mock_emp2.identifier = 20
        self.mock_emp2.calculate_pay_month.return_value = 2500.0

    def test_add_and_is_employee(self):
        self.assertFalse(self.payroll.is_employee(self.mock_emp1))
        self.payroll.add_employee(self.mock_emp1)
        self.assertTrue(self.payroll.is_employee(self.mock_emp1))

        # Adding duplicate should raise ValueError
        with self.assertRaises(ValueError):
            self.payroll.add_employee(self.mock_emp1)

    def test_remove_employee(self):
        self.payroll.add_employee(self.mock_emp1)
        self.payroll.remove_employee(self.mock_emp1)
        self.assertFalse(self.payroll.is_employee(self.mock_emp1))

        # Removing non-existent should raise ValueError
        with self.assertRaises(ValueError):
            self.payroll.remove_employee(self.mock_emp1)

    def test_get_employee(self):
        self.payroll.add_employee(self.mock_emp1)
        retrieved = self.payroll.get_employee(10)
        self.assertEqual(retrieved, self.mock_emp1)

    def test_calculate_pay_month_and_payroll(self):
        self.payroll.add_employee(self.mock_emp1)
        self.payroll.add_employee(self.mock_emp2)

        expected_dict = {10: 1500.0, 20: 2500.0}
        self.assertEqual(self.payroll.calculate_pay_month(), expected_dict)
        self.assertAlmostEqual(self.payroll.calculate_payroll(), 4000.0)

    def test_generate_identifier(self):
        self.assertEqual(self.payroll.generate_identifier(), 1)
        self.payroll.add_employee(self.mock_emp1)  # ID is 10
        self.assertEqual(self.payroll.generate_identifier(), 11)


Overwriting code/exercise3/tests/test_PayrollSystem.py


In [59]:
!cd code/exercise3 && python3 -m unittest discover -s tests
!python3 -m mypy --strict code/exercise3/api

..........
----------------------------------------------------------------------
Ran 10 tests in 0.003s

OK
Success: no issues found in 6 source files


## Exercise 4 (optional) -- the same suite under `pytest`

**Do this once exercises 1 to 3 are written**, otherwise there is nothing to run.

The closing section of Lecture 3 claims that `pytest` runs `unittest.TestCase` classes unchanged. Check it.
Install it once (`pip install pytest`), then run the next cell on the tests *you* have just written: the same tests,
the same failures, and rather more informative messages. (If it reports `no tests ran`, you have not written any yet.)

In [62]:
%pip install pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]2m2/3 [pytest]
Note: you may need to restart the kernel to use updated packages.


In [63]:
%%writefile code/exercise3/tests/test_SalaryEmployee.py
import pytest
from api import SalaryEmployee


@pytest.fixture
def sample_salary_employee() -> SalaryEmployee:
    return SalaryEmployee(1, "Bob", 120000.0)


def test_init_properties(sample_salary_employee: SalaryEmployee) -> None:
    assert sample_salary_employee.identifier == 1
    assert sample_salary_employee.name == "Bob"


def test_calculate_pay_month(sample_salary_employee: SalaryEmployee) -> None:
    assert sample_salary_employee.calculate_pay_month() == 10000.0


Overwriting code/exercise3/tests/test_SalaryEmployee.py


In [64]:
!cd code/exercise3 && python3 -m pytest -v

============================= test session starts ==============================
platform darwin -- Python 3.15.0, pytest-9.1.1, pluggy-1.6.0 -- /usr/local/bin/python3
cachedir: .pytest_cache
rootdir: /Users/hoangtom/Documents/CODE/USTH_ProgrammingTechniques/Week1/Labworks/Labwork3/code/exercise3
plugins: platformdirs-4.13.0
collected 11 items                                                             

tests/test_CommissionEmployee.py::TestCommissionEmployee::test_init_and_calculation PASSED [  9%]
tests/test_Employee.py::TestEmployee::test_cannot_instantiate_abstract_class PASSED [ 18%]
tests/test_Employee.py::TestEmployee::test_concrete_subclass_properties PASSED [ 27%]
tests/test_HourlyEmployee.py::TestHourlyEmployee::test_hours_worked_and_calculation PASSED [ 36%]
tests/test_PayrollSystem.py::TestPayrollSystem::test_add_and_is_employee PASSED [ 45%]
tests/test_PayrollSystem.py::TestPayrollSystem::test_calculate_pay_month_and_payroll PASSED [ 54%]
tests/test_PayrollSystem.py::Test

Then pick **one** test file -- `tests/test_SalaryEmployee.py` is the shortest -- and rewrite it in `pytest` style:
plain `test_*` functions instead of a `TestCase`, bare `assert` instead of `assertEqual`, `pytest.raises` instead of
`assertRaises`, and a `@pytest.fixture` for the employee you build in every test. Keep the other files as they are and
run the suite again: the two styles coexist in the same directory, which is exactly how a real project migrates.

Note that `unittest.mock` does not change at all -- `patch`, `Mock` and `MagicMock` are standard library, and exercise 3
above uses them identically whichever runner you choose.